# Workflow Setup

Import order:
1. Utils
2. Nodes
3. Graph

# **Define Utilities**

Required Libraries


In [1]:
import json
import re
from typing import Annotated, Any, Dict, NotRequired, TypedDict

import yaml
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

**DSRP STATE**



In [2]:
def merge_dicts(left: Dict[str, Any], right: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(left or {})
    merged.update(right or {})
    return merged


class DSRPState(TypedDict):
    paper_id: str
    dsrp_outputs: Annotated[Dict[str, Any], merge_dicts]
    collection_name: str
    persist_directory: str
    embedding_model: str
    gatekeeper: NotRequired[Dict[str, Any]]

**LLM**

In [3]:
def set_llm():
    return ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
    )

**Prompt Loading Functions**


In [4]:
def load_vector_query(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    if "query" not in config:
        raise ValueError(f"'query' key not found in {path}")

    return {
        "query": config["query"],
        "k": config.get("k", 10),
        "name": config.get("name", None),
        "description": config.get("description", None),
    }


def load_yaml_prompt(path: str):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    return ChatPromptTemplate.from_messages(
        [
            ("system", config["instructions"]),
            ("human", "{input}"),
        ],
        template_format="jinja2",
    )


**Retriever**

In [5]:
class PaperRetriever:
    def __init__(self, collection_name, persist_directory, embedding_model):
        self._vectorstore = Chroma(
            collection_name=collection_name,
            embedding_function=OpenAIEmbeddings(model=embedding_model),
            persist_directory=persist_directory,
        )

    def for_paper(self, paper_id: str, k: int = 6):
        return self._vectorstore.as_retriever(
            search_kwargs={
                "k": k,
                "filter": {"paper_id": paper_id},
            }
        )

LLM Response Parser

In [6]:
def parse_llm_json(text: str) -> dict:
    s = (text or "").strip()
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE | re.DOTALL).strip()

    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass

    s2 = re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", s)

    try:
        return json.loads(s2)
    except json.JSONDecodeError as e:
        raise ValueError(f"LLM returned invalid JSON: {e}\nRaw:\n{s[:1200]}")

# **Define LangGraph Nodes**

Gatekeeper Node

In [7]:
def gatekeeper_node(state: DSRPState) -> dict:
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]

    vector_config = load_vector_query("prompts/ds_gatekeeper/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt("prompts/ds_gatekeeper/retriever.yaml")
    llm = set_llm()

    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = json.loads(evidence_response.content)

    classifier_prompt = load_yaml_prompt("prompts/ds_gatekeeper/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = json.loads(classification_response.content)

    auditor_prompt = load_yaml_prompt("prompts/ds_gatekeeper/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = json.loads(audit_response.content)

    return {"gatekeeper": audit_json}

**Research Question Node**

In [8]:
def research_question_node(state: DSRPState):
    dimension_name = "research_question"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]

    vector_config = load_vector_query(f"prompts/dsrp/{dimension_name}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/retriever.yaml")
    llm = set_llm()

    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"][dimension_name] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}



**Data Understanding Node**


In [9]:
def data_understanding_node(state: DSRPState):
    dimension_name = "data_understanding"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]

    vector_config = load_vector_query(f"prompts/dsrp/{dimension_name}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/retriever.yaml")
    llm = set_llm()

    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"][dimension_name] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}



**Data Pre-processing Node**

In [10]:
def data_preprocessing_node(state: DSRPState):
    dimension_name = "data_preprocessing"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]

    vector_config = load_vector_query(f"prompts/dsrp/{dimension_name}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/retriever.yaml")
    llm = set_llm()

    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"prompts/dsrp/{dimension_name}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"][dimension_name] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}



**Modelling Paradigms Node**

In [11]:
def modelling_node(state: DSRPState):
    base_path = "prompts/dsrp/modelling"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    foundational_path = f"{base_path}/foundational"
    specialised_path = f"{base_path}/specialised"

    vector_config = load_vector_query(f"{foundational_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{foundational_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    evidence_json["modelling_evidence"] = [
        e for e in evidence_json["modelling_evidence"] if e.get("method_used", True)
    ]

    foundational_prompt = load_yaml_prompt(f"{foundational_path}/foundational_classifier.yaml")
    foundational_response = llm.invoke(
        foundational_prompt.format_messages(input=json.dumps(evidence_json))
    )
    foundational_json = parse_llm_json(foundational_response.content)

    ml_json = {}
    if foundational_json.get("foundational_paradigm") in ["Machine Learning", "Hybrid / Integrated"]:
        ml_prompt = load_yaml_prompt(f"{foundational_path}/ml_learning_classifier.yaml")
        ml_response = llm.invoke(ml_prompt.format_messages(input=json.dumps(evidence_json)))
        ml_json = parse_llm_json(ml_response.content)

    foundational_audit_prompt = load_yaml_prompt(f"{foundational_path}/auditor.yaml")
    foundational_audit_response = llm.invoke(
        foundational_audit_prompt.format_messages(
            input=json.dumps(
                {
                    "foundational": foundational_json,
                    "ml_details": ml_json,
                    "evidence": evidence_json,
                }
            )
        )
    )
    foundational_audit_json = parse_llm_json(foundational_audit_response.content)

    vector_config = load_vector_query(f"{specialised_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    specialised_context = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{specialised_path}/retriever.yaml")
    specialised_evidence_response = llm.invoke(
        retriever_prompt.format_messages(input=specialised_context)
    )
    specialised_evidence_json = parse_llm_json(specialised_evidence_response.content)

    specialised_evidence_json["modelling_evidence"] = [
        e for e in specialised_evidence_json["modelling_evidence"] if e.get("method_used", True)
    ]

    specialised_prompt = load_yaml_prompt(f"{specialised_path}/specialised_classifier.yaml")
    specialised_response = llm.invoke(
        specialised_prompt.format_messages(input=json.dumps(specialised_evidence_json))
    )
    specialised_json = parse_llm_json(specialised_response.content)

    sub_json = {}
    if specialised_json.get("specialised_paradigms"):
        sub_prompt = load_yaml_prompt(f"{specialised_path}/sub_specialised_classifier.yaml")
        sub_response = llm.invoke(
            sub_prompt.format_messages(
                input=json.dumps(
                    {
                        "evidence": specialised_evidence_json,
                        "specialised": specialised_json,
                    }
                )
            )
        )
        sub_json = parse_llm_json(sub_response.content)

    specialised_audit_prompt = load_yaml_prompt(f"{specialised_path}/auditor.yaml")
    specialised_audit_response = llm.invoke(
        specialised_audit_prompt.format_messages(
            input=json.dumps(
                {
                    "specialised": specialised_json,
                    "sub_paradigms": sub_json,
                    "evidence": specialised_evidence_json,
                }
            )
        )
    )
    specialised_audit_json = parse_llm_json(specialised_audit_response.content)

    global_audit_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_response = llm.invoke(
        global_audit_prompt.format_messages(
            input=json.dumps(
                {
                    "foundational": foundational_audit_json,
                    "specialised": specialised_audit_json,
                }
            )
        )
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["modelling"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}



**Evaluation Nodes**

In [12]:


def evaluation_metrics_foundational_node(state: DSRPState):
    base_path = "prompts/dsrp/evaluation/metrics/foundational"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    vector_config = load_vector_query(f"{base_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{base_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    modelling_output = state["dsrp_outputs"].get("modelling", {})
    foundational_paradigm = modelling_output.get("foundational_paradigm")
    ml_learning_type = modelling_output.get("ml_learning_type")

    classifier_prompt = load_yaml_prompt(f"{base_path}/classifier.yaml")
    classifier_input = {
        "modelling_context": {
            "foundational_paradigm": foundational_paradigm,
            "ml_learning_type": ml_learning_type,
        },
        "evidence": evidence_json,
    }

    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(classifier_input))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_input = {
        "modelling_context": {
            "foundational_paradigm": foundational_paradigm,
            "ml_learning_type": ml_learning_type,
        },
        "classification": classification_json,
    }

    audit_response = llm.invoke(auditor_prompt.format_messages(input=json.dumps(audit_input)))
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["evaluation_metrics_foundational"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}


def evaluation_metrics_specialised_node(state: DSRPState):
    base_path = "prompts/dsrp/evaluation/metrics/specialised"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    modelling_output = state["dsrp_outputs"].get("modelling", {})
    specialised_paradigms = modelling_output.get("specialised_paradigms", [])

    if not specialised_paradigms:
        state["dsrp_outputs"]["evaluation_metrics_specialised"] = {
            "specialised_metrics_reported": [],
            "evaluation_quality": "Not applicable",
            "confidence": 1.0,
            "validated_reasoning": "No specialised analytical paradigms were detected in the modelling stage.",
            "validated_bibliography": [],
            "audit_commentary": "",
        }
        return {"dsrp_outputs": state["dsrp_outputs"]}

    vector_config = load_vector_query(f"{base_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{base_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"{base_path}/classifier.yaml")
    classifier_input = {
        "specialised_paradigms": specialised_paradigms,
        "evidence": evidence_json,
    }

    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(classifier_input))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_input = {
        "specialised_paradigms": specialised_paradigms,
        "classification": classification_json,
    }

    audit_response = llm.invoke(auditor_prompt.format_messages(input=json.dumps(audit_input)))
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["evaluation_metrics_specialised"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}


def evaluation_theoretical_orientation_node(state: DSRPState):
    base_path = "prompts/dsrp/evaluation/epistemological/theoretical_orientation"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    vector_config = load_vector_query(f"{base_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{base_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"{base_path}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["evaluation_theoretical_orientation"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}


def evaluation_interpretability_node(state: DSRPState):
    base_path = "prompts/dsrp/evaluation/epistemological/interpretability"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    vector_config = load_vector_query(f"{base_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{base_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"{base_path}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["evaluation_interpretability"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}


def evaluation_ethical_social_node(state: DSRPState):
    base_path = "prompts/dsrp/evaluation/ethical_social"
    collection_name = state["collection_name"]
    persist_directory = state["persist_directory"]
    embedding_model = state["embedding_model"]
    llm = set_llm()

    vector_config = load_vector_query(f"{base_path}/vector_query.yaml")

    retriever_tool = PaperRetriever(
        collection_name=collection_name,
        persist_directory=persist_directory,
        embedding_model=embedding_model,
    ).for_paper(state["paper_id"], k=vector_config["k"])

    docs = retriever_tool.invoke(vector_config["query"])

    context_text = "\n\n".join(
        f"[Page {d.metadata.get('page_no')} | {d.metadata.get('section_heading')}]\n{d.page_content}"
        for d in docs
    )

    retriever_prompt = load_yaml_prompt(f"{base_path}/retriever.yaml")
    evidence_response = llm.invoke(retriever_prompt.format_messages(input=context_text))
    evidence_json = parse_llm_json(evidence_response.content)

    classifier_prompt = load_yaml_prompt(f"{base_path}/classifier.yaml")
    classification_response = llm.invoke(
        classifier_prompt.format_messages(input=json.dumps(evidence_json))
    )
    classification_json = parse_llm_json(classification_response.content)

    auditor_prompt = load_yaml_prompt(f"{base_path}/auditor.yaml")
    audit_response = llm.invoke(
        auditor_prompt.format_messages(input=json.dumps(classification_json))
    )
    audit_json = parse_llm_json(audit_response.content)

    state["dsrp_outputs"]["evaluation_ethical_social"] = audit_json
    return {"dsrp_outputs": state["dsrp_outputs"]}

**LanghGraph Architecture**

Parallel

In [13]:
# Build mixed graph: parallel branches + dependent chains
from langgraph.graph import END, StateGraph


def route_after_gatekeeper(state: DSRPState):
    if state["gatekeeper"]["final_classification"] == "Exclude":
        return "end"
    return "continue"


def fanout_node(state: DSRPState):
    # No-op node used to start multiple branches in parallel.
    return {}


builder = StateGraph(DSRPState)

builder.add_node("gatekeeper", gatekeeper_node)
builder.add_node("fanout", fanout_node)

builder.add_node("research_question", research_question_node)
builder.add_node("data_understanding", data_understanding_node)
builder.add_node("data_preprocessing", data_preprocessing_node)

builder.add_node("modelling", modelling_node)
builder.add_node("evaluation_metrics_foundational_node", evaluation_metrics_foundational_node)
builder.add_node("evaluation_metrics_specialised_node", evaluation_metrics_specialised_node)

builder.add_node("evaluation_theoretical_orientation_node", evaluation_theoretical_orientation_node)
builder.add_node("evaluation_interpretability_node", evaluation_interpretability_node)
builder.add_node("evaluation_ethical_social_node", evaluation_ethical_social_node)

builder.set_entry_point("gatekeeper")

# Gatekeeper decides whether to stop or continue into parallel fan-out.
builder.add_conditional_edges(
    "gatekeeper",
    route_after_gatekeeper,
    {
        "end": END,
        "continue": "fanout",
    },
)

# Parallel branch starts
builder.add_edge("fanout", "research_question")
builder.add_edge("fanout", "modelling")
builder.add_edge("fanout", "evaluation_theoretical_orientation_node")
builder.add_edge("fanout", "evaluation_interpretability_node")
builder.add_edge("fanout", "evaluation_ethical_social_node")

# Dependent chain A
builder.add_edge("research_question", "data_understanding")
builder.add_edge("data_understanding", "data_preprocessing")
builder.add_edge("data_preprocessing", END)

# Dependent chain B
builder.add_edge("modelling", "evaluation_metrics_foundational_node")
builder.add_edge("evaluation_metrics_foundational_node", "evaluation_metrics_specialised_node")
builder.add_edge("evaluation_metrics_specialised_node", END)

# Independent branch ends
builder.add_edge("evaluation_theoretical_orientation_node", END)
builder.add_edge("evaluation_interpretability_node", END)
builder.add_edge("evaluation_ethical_social_node", END)

graph = builder.compile()

# Run

In [14]:
COLLECTION_NAME = "theory_synthesis"
PERSIST_DIRECTORY = "./vector_database"
embedding_model = "text-embedding-3-small"
PAPER_FOLDER = "./my_papers"

In [15]:
test_state: DSRPState = {
    "paper_id": "2024 - 15.pdf",
    "dsrp_outputs": {},
    "collection_name": COLLECTION_NAME,
    "persist_directory": PERSIST_DIRECTORY,
    "embedding_model": embedding_model
}

result = graph.invoke(test_state)

print(result)

C:\Users\sahil\AppData\Local\Temp\ipykernel_16172\2525832046.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


{'paper_id': '2024 - 15.pdf', 'dsrp_outputs': {'evaluation_theoretical_orientation': {'explicit_theory': '', 'implicit_theory_detected': '', 'data_science_role': '', 'research_motivation': '', 'contribution_type': '', 'confidence': 0.0, 'validated_reasoning': '', 'validated_bibliography': [], 'audit_commentary': ''}, 'evaluation_interpretability': {'interpretability_discussed': '', 'interpretability_method': '', 'model_transparency_level': '', 'confidence': 0.0, 'validated_reasoning': '', 'validated_bibliography': [], 'audit_commentary': 'No input provided to validate interpretability claims.'}, 'research_question': {'final_classification': '', 'confidence': 0.0, 'validated_reasoning': '', 'validated_bibliography': [{'id': 1, 'page': '', 'section': '', 'direct_quote': ''}], 'audit_commentary': 'No input provided for validation.'}, 'evaluation_ethical_social': {'privacy_protection_reported': '', 'bias_fairness_considered': '', 'societal_impact_discussed': '', 'ethical_reflection_level':

In [16]:
from html import escape
from IPython.display import HTML, display

cards = []
for node, payload in result.get("dsrp_outputs", {}).items():
    confidence = payload.get("confidence", "-")
    summary = (
        payload.get("final_classification")
        or payload.get("evaluation_quality")
        or payload.get("foundational_paradigm")
        or ""
    )
    cards.append(
        f"""
        <details class='card'>
          <summary><b>{escape(node)}</b> | confidence: {escape(str(confidence))} {escape(str(summary))}</summary>
          <pre>{escape(json.dumps(payload, indent=2, ensure_ascii=False))}</pre>
        </details>
        """
    )

display(HTML(
    """
    <style>
      .grid {display:grid; gap:10px;}
      .card {border:1px solid #d7d7d7; border-radius:8px; padding:8px 10px; background:#fafafa;}
      summary {cursor:pointer;}
      pre {white-space:pre-wrap; margin:8px 0 0; font-size:12px;}
    </style>
    <h3>DSRP Results</h3>
    <div class='grid'>""" + "".join(cards) + "</div>"
))